# 06 — Crude Oil Correlation Analysis

This is the core analysis: does daily sentiment correlate with crude oil prices?

| Section | Content |
|---|---|
| 1 | Aggregate sentiment to daily scores |
| 2 | Merge with crude oil prices |
| 3 | Pearson correlation table |
| 4 | Sentiment vs oil price (dual-axis) |
| 5 | Correlation scatter plots |
| 6 | 7-day rolling correlation |
| 7 | Interpretation and conclusions |

> **Final notebook — see README for complete results.**

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib
matplotlib.use('Agg')
from src.config import SENTIMENT_RESULTS_FILE, CRUDE_RAW_FILE, CORRELATION_FILE

sent_df  = pd.read_csv(SENTIMENT_RESULTS_FILE)
crude_df = pd.read_csv(CRUDE_RAW_FILE)
print(f'Sentiment rows : {len(sent_df)}')
print(f'Crude oil rows : {len(crude_df)}')

## 1. Daily Sentiment Aggregation

Average compound scores per calendar day. Only calendar days with articles contribute a data point.

In [ ]:
from src.sentiment import aggregate_daily

daily_sentiment = aggregate_daily(sent_df)
print(f'Daily aggregation: {len(daily_sentiment)} days')
daily_sentiment.head(10)

## 2. Merge with Crude Oil Prices

Inner join on date — only trading days with matching sentiment data are kept.

In [ ]:
daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date']).dt.date
crude_df['date']        = pd.to_datetime(crude_df['date']).dt.date

merged = pd.merge(daily_sentiment, crude_df, on='date', how='inner')
print(f'Merged dataset: {len(merged)} days')
print(f'Sentiment days: {len(daily_sentiment)}')
print(f'Oil trading days: {len(crude_df)}')
merged[['date', 'vader_compound', 'Brent_Close', 'WTI_Close']].head(10)

## 3. Pearson Correlation Table

In [ ]:
print('Correlation Analysis:')
print('-' * 60)

pairs = [('vader_compound', 'VADER', 'Brent_Close', 'Brent'),
         ('vader_compound', 'VADER', 'WTI_Close', 'WTI')]
if 'roberta_compound' in merged.columns:
    pairs += [('roberta_compound', 'RoBERTa', 'Brent_Close', 'Brent'),
              ('roberta_compound', 'RoBERTa', 'WTI_Close', 'WTI')]

for sent_col, sent_name, price_col, market in pairs:
    sub = merged[[sent_col, price_col]].dropna()
    r   = sub[sent_col].corr(sub[price_col])
    strength = 'Strong' if abs(r) >= 0.5 else 'Moderate' if abs(r) >= 0.3 else 'Weak'
    direction = 'positive' if r > 0 else 'negative'
    print(f'  {sent_name:10s} vs {market:5s}: r = {r:.4f}  ({strength} {direction})')

In [ ]:
# Load saved correlation results
if CORRELATION_FILE.exists():
    corr_df = pd.read_csv(CORRELATION_FILE)
    print('Full correlation table:')
    print(corr_df.to_string(index=False))

## 4. Sentiment vs Oil Price (Dual-Axis)

In [ ]:
from src.visualize import plot_sentiment_vs_oil

plot_sentiment_vs_oil(merged)
print('Saved: images/09_sentiment_vs_oil_price.png')

## 5. Correlation Scatter Plots

In [ ]:
from src.visualize import plot_correlation_scatter

plot_correlation_scatter(merged)
print('Saved: images/10_correlation_scatter.png')

## 6. 7-Day Rolling Correlation

How does the correlation strength vary over time? Peaks correspond to escalation events, troughs to diplomatic pauses.

In [ ]:
from src.visualize import plot_rolling_correlation

plot_rolling_correlation(merged)
print('Saved: images/11_rolling_correlation.png')

## 7. Interpretation

### Results

| Model | Market | Pearson r | Interpretation |
|---|---|---|---|
| VADER | Brent | -0.4646 | Moderate negative |
| VADER | WTI | -0.4840 | Moderate negative |
| RoBERTa | Brent | -0.2962 | Weak negative |
| RoBERTa | WTI | -0.2770 | Weak negative |

### What does negative correlation mean here?

When VADER compound is **more positive** (lower negativity intensity in war news), oil prices tend to be **higher**. This is consistent with:

- **Easing tension** → reduced supply risk perception → higher demand confidence → higher prices (or the inverse: escalation → demand uncertainty → price suppression)
- War-related demand destruction fears may dominate supply-disruption premiums in the current period

### VADER vs RoBERTa gap

VADER detects stronger signal (r = -0.46 vs -0.30) because it responds to raw vocabulary — the same signal commodity markets embed in price. RoBERTa's neutral classification of objective journalism reduces its correlation strength.

> **See `reports/War_Sentiment_Process_Report.pdf` for the full analysis writeup.**